In [5]:
!pip install flask flask-cors pyngrok --quiet

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.utils import to_categorical
from flask import Flask, request, jsonify, render_template_string
from flask_cors import CORS
from pyngrok import ngrok
import base64
import io
from PIL import Image

# 1. Train Model (MNIST)
mnist = tf.keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Training CNN Model...")
model.fit(X_train, y_train, epochs=3, batch_size=64)

# 2. Setup Web Server UI
app = Flask(__name__)
CORS(app)

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>Handwritten Character Recognition</title>
    <style>
        body { font-family: Arial, sans-serif; text-align: center; margin-top: 40px; background-color: #f4f4f9; }
        #canvas { border: 3px solid #333; background-color: black; cursor: crosshair; border-radius: 8px; }
        button { padding: 10px 25px; font-size: 16px; margin: 10px; cursor: pointer; border: none; border-radius: 5px; }
        .btn-predict { background-color: #ff6f00; color: white; }
        .btn-clear { background-color: #777; color: white; }
        #result { font-size: 28px; font-weight: bold; margin-top: 20px; color: #222; }
    </style>
</head>
<body>
    <h2>Handwritten Character Recognition</h2>
    <p>Draw a digit (0-9) inside the black box:</p>
    <canvas id="canvas" width="280" height="280"></canvas><br>
    <button class="btn-clear" onclick="clearCanvas()">Clear</button>
    <button class="btn-predict" onclick="predict()">Predict</button>
    <div id="result">Prediction: --</div>

    <script>
        const canvas = document.getElementById('canvas');
        const ctx = canvas.getContext('2d');
        let drawing = false;

        ctx.strokeStyle = "white";
        ctx.lineWidth = 18;
        ctx.lineCap = "round";

        canvas.onmousedown = () => drawing = true;
        canvas.onmouseup = () => { drawing = false; ctx.beginPath(); };
        canvas.onmousemove = draw;

        function draw(e) {
            if (!drawing) return;
            const rect = canvas.getBoundingClientRect();
            ctx.lineTo(e.clientX - rect.left, e.clientY - rect.top);
            ctx.stroke();
            ctx.beginPath();
            ctx.moveTo(e.clientX - rect.left, e.clientY - rect.top);
        }

        function clearCanvas() {
            ctx.fillRect(0, 0, canvas.width, canvas.height);
            ctx.fillStyle = "black";
            ctx.fillRect(0, 0, canvas.width, canvas.height);
            document.getElementById('result').innerText = "Prediction: --";
        }
        clearCanvas();

        function predict() {
            const dataURL = canvas.toDataURL('image/png');
            fetch('/predict', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ image: dataURL })
            })
            .then(res => res.json())
            .then(data => {
                document.getElementById('result').innerText = `Prediction: ${data.digit} (${(data.confidence * 100).toFixed(1)}% Confidence)`;
            });
        }
    </script>
</body>
</html>
"""

@app.route('/')
def home():
    return render_template_string(HTML_PAGE)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json['image']
    header, encoded = data.split(",", 1)
    image_data = base64.b64decode(encoded)

    img = Image.open(io.BytesIO(image_data)).convert('L')
    img = img.resize((28, 28))
    img_array = np.array(img).reshape(1, 28, 28, 1).astype('float32') / 255.0

    preds = model.predict(img_array)[0]
    predicted_digit = int(np.argmax(preds))
    confidence = float(np.max(preds))

    return jsonify({'digit': predicted_digit, 'confidence': confidence})

# Run Web Server via Google Colab Port Forwarding
from google.colab.output import eval_js
print("\n--- SERVER READY ---")
print("Click the URL below to open your web page:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

app.run(port=5000)

Training CNN Model...
Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 49s 50ms/step - accuracy: 0.9455 - loss: 0.1768
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 48s 52ms/step - accuracy: 0.9833 - loss: 0.0553
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 47s 50ms/step - accuracy: 0.9875 - loss: 0.0401

--- SERVER READY ---
Click the URL below to open your web page:
https://5000-m-s-kkb-usw4a0-tzqumd3rdm25-a.us-west4-0.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [31/Aug/2026 20:12:28] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Aug/2026 20:12:30] "GET /favicon.ico HTTP/1.1" 404 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step


INFO:werkzeug:127.0.0.1 - - [31/Aug/2026 20:12:43] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


...

INFO:werkzeug:127.0.0.1 - - [31/Aug/2026 20:14:34] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step


INFO:werkzeug:127.0.0.1 - - [31/Aug/2026 20:14:35] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


INFO:werkzeug:127.0.0.1 - - [31/Aug/2026 20:14:43] "POST /predict HTTP/1.1" 200 -
